In [5]:
import os
import glob
import math
from typing import Dict, Tuple, Optional, List

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt


# =========================
# Paths / Settings
# =========================
BASE_DIR = "."                 # 현재 폴더(= 4_DAG_visualization)에서 실행 기준
OUT_BASE = "./layouts"         # 결과 저장 폴더

FIGSIZE = (14, 10)
DPI = 220

NODE_SIZE = 180
EDGE_WIDTH = 0.8
ARROWSIZE = 10

WITH_LABELS = False
LABEL_FONT_SIZE = 7
MAX_LABEL_NODES = 60

SEED = 42


# =========================
# IO
# =========================
def ensure_dir(p: str) -> None:
    os.makedirs(p, exist_ok=True)


def alg_name_from_filename(path: str) -> str:
    base = os.path.basename(path)
    name = os.path.splitext(base)[0]
    return name[6:] if name.lower().startswith("graph_") else name


def load_gexf_as_digraph(path: str) -> nx.DiGraph:
    """
    gexf가 MultiDiGraph로 들어오거나 노드 id 타입이 섞여도,
    최종적으로 '문자열 노드 id'의 DiGraph로 고정.
    """
    G0 = nx.read_gexf(path)

    # MultiDiGraph -> DiGraph (가중치 합/마지막 등 정책 필요)
    if isinstance(G0, nx.MultiDiGraph):
        G = nx.DiGraph()
        for n, data in G0.nodes(data=True):
            G.add_node(str(n), **data)
        for u, v, key, data in G0.edges(keys=True, data=True):
            u2, v2 = str(u), str(v)
            w = data.get("weight", 1.0)
            try:
                w = float(w)
            except Exception:
                w = 1.0
            # 중복 edge면 weight 누적(원하면 max로 바꿀 수 있음)
            if G.has_edge(u2, v2):
                G[u2][v2]["weight"] = float(G[u2][v2].get("weight", 0.0)) + w
            else:
                G.add_edge(u2, v2, weight=w)
    else:
        G = nx.DiGraph()
        for n, data in G0.nodes(data=True):
            G.add_node(str(n), **data)
        for u, v, data in G0.edges(data=True):
            u2, v2 = str(u), str(v)
            w = data.get("weight", 1.0)
            try:
                w = float(w)
            except Exception:
                w = 1.0
            G.add_edge(u2, v2, weight=w)

    # diag/self-loop 제거(있으면)
    self_loops = list(nx.selfloop_edges(G))
    if self_loops:
        G.remove_edges_from(self_loops)

    return G


# =========================
# Pos utilities (중요: 누락 보정)
# =========================
def normalize_pos(G: nx.DiGraph, pos: Dict) -> Dict:
    """
    어떤 레이아웃이든 pos가 일부 노드만 포함하는 경우가 있어
    draw 단계에서 KeyError가 납니다.
    -> 누락 노드는 circular 좌표로 채워서 항상 '모든 노드 pos' 보장.
    """
    if pos is None:
        pos = {}

    # keys를 str로 정리
    pos2 = {str(k): (float(v[0]), float(v[1])) for k, v in pos.items()}

    missing = [n for n in G.nodes() if n not in pos2]
    if missing:
        # fallback: circular layout for missing nodes (전체에 대해 만들고 필요한 것만 사용)
        fallback = nx.circular_layout(G)
        for n in missing:
            xy = fallback.get(n, (0.0, 0.0))
            pos2[n] = (float(xy[0]), float(xy[1]))

    # NaN/inf 방지
    for n, (x, y) in list(pos2.items()):
        if not np.isfinite(x) or not np.isfinite(y):
            pos2[n] = (0.0, 0.0)

    return pos2


def to_undirected_weighted(G: nx.DiGraph) -> nx.Graph:
    H = nx.Graph()
    H.add_nodes_from(G.nodes())
    for u, v, data in G.edges(data=True):
        w = float(data.get("weight", 1.0))
        H.add_edge(u, v, weight=abs(w))
    return H


# =========================
# Layouts (10)
# =========================
def layout_spring_fr(G: nx.DiGraph) -> Dict:
    H = to_undirected_weighted(G)
    return nx.spring_layout(H, seed=SEED, weight="weight")


def layout_kamada_kawai(G: nx.DiGraph) -> Dict:
    H = to_undirected_weighted(G)
    return nx.kamada_kawai_layout(H, weight="weight")


def hierarchical_fallback(G: nx.DiGraph) -> Dict:
    """
    DAG이면 위상정렬 기반 layer.
    DAG이 아니면: 임의 order로 forward edge만 남겨 근사 DAG 만든 후 layer.
    """
    H = G.copy()
    if not nx.is_directed_acyclic_graph(H):
        nodes = list(H.nodes())
        order = {n: i for i, n in enumerate(nodes)}
        H2 = nx.DiGraph()
        H2.add_nodes_from(nodes)
        for u, v, data in H.edges(data=True):
            if order[u] < order[v]:
                H2.add_edge(u, v, **data)
        H = H2

    try:
        topo = list(nx.topological_sort(H))
    except Exception:
        topo = list(H.nodes())

    layer = {n: 0 for n in topo}
    for n in topo:
        preds = list(H.predecessors(n))
        if preds:
            layer[n] = max(layer[p] + 1 for p in preds)

    buckets: Dict[int, List[str]] = {}
    for n, l in layer.items():
        buckets.setdefault(l, []).append(n)

    max_layer = max(buckets.keys()) if buckets else 0
    pos = {}
    for l in range(max_layer + 1):
        ns = sorted(buckets.get(l, []))
        k = len(ns)
        if k == 0:
            continue
        for i, n in enumerate(ns):
            x = (i + 1) / (k + 1)
            y = -float(l)
            pos[n] = (x, y)
    return pos


def layout_hierarchical(G: nx.DiGraph) -> Tuple[Dict, str]:
    try:
        from networkx.drawing.nx_agraph import graphviz_layout
        return graphviz_layout(G, prog="dot"), "dot"
    except Exception:
        return hierarchical_fallback(G), "fallback"


def layout_tree(G: nx.DiGraph) -> Tuple[Dict, str]:
    nodes = list(G.nodes())
    if not nodes:
        return {}, "empty"

    sources = [n for n in nodes if G.in_degree(n) == 0]
    if sources:
        root = max(sources, key=lambda n: G.out_degree(n))
    else:
        root = max(nodes, key=lambda n: G.out_degree(n))

    T = nx.bfs_tree(G, source=root)
    try:
        from networkx.drawing.nx_agraph import graphviz_layout
        return graphviz_layout(T, prog="dot"), "dot"
    except Exception:
        return hierarchical_fallback(T), "fallback"


def layout_circular(G: nx.DiGraph) -> Dict:
    return nx.circular_layout(G)


def layout_shell_concentric(G: nx.DiGraph) -> Dict:
    nodes = list(G.nodes())
    if not nodes:
        return {}
    indeg = {n: G.in_degree(n) for n in nodes}
    sorted_nodes = sorted(nodes, key=lambda n: indeg[n], reverse=True)

    n = len(sorted_nodes)
    s1 = sorted_nodes[: max(1, int(0.10 * n))]
    s2 = sorted_nodes[max(1, int(0.10 * n)) : max(2, int(0.40 * n))]
    s3 = sorted_nodes[max(2, int(0.40 * n)) :]
    shells = [s1, s2, s3]
    H = to_undirected_weighted(G)
    return nx.shell_layout(H, nlist=shells)


def layout_forceatlas2(G: nx.DiGraph) -> Tuple[Dict, str]:
    H = to_undirected_weighted(G)
    try:
        from fa2 import ForceAtlas2  # pip install fa2
        nodes = list(H.nodes())
        idx = {n: i for i, n in enumerate(nodes)}

        try:
            A = nx.to_scipy_sparse_array(H, nodelist=nodes, weight="weight", dtype=float)
        except Exception:
            A = nx.to_numpy_array(H, nodelist=nodes, weight="weight", dtype=float)

        fa = ForceAtlas2(
            outboundAttractionDistribution=True,
            linLogMode=False,
            adjustSizes=False,
            edgeWeightInfluence=1.0,
            jitterTolerance=1.0,
            barnesHutOptimize=True,
            barnesHutTheta=1.2,
            scalingRatio=2.0,
            strongGravityMode=False,
            gravity=1.0,
            verbose=False,
        )
        coords = fa.forceatlas2(A, pos=None, iterations=200)
        pos = {n: (float(coords[idx[n], 0]), float(coords[idx[n], 1])) for n in nodes}
        return pos, "fa2"
    except Exception:
        return nx.spring_layout(H, seed=SEED, weight="weight"), "fallback_spring"


def layout_yifan_hu(G: nx.DiGraph) -> Tuple[Dict, str]:
    H = to_undirected_weighted(G)
    try:
        import igraph as ig  # pip install igraph
        nodes = list(H.nodes())
        idx = {n: i for i, n in enumerate(nodes)}
        edges = [(idx[u], idx[v]) for u, v in H.edges()]
        weights = [float(H[u][v].get("weight", 1.0)) for u, v in H.edges()]

        g = ig.Graph(n=len(nodes), edges=edges, directed=False)
        g.es["weight"] = weights
        lay = g.layout("yifanhu", weights="weight")
        pos = {nodes[i]: (float(lay[i][0]), float(lay[i][1])) for i in range(len(nodes))}
        return pos, "igraph"
    except Exception:
        k = 1.0 / math.sqrt(max(1, H.number_of_nodes()))
        return nx.spring_layout(H, seed=SEED, weight="weight", iterations=100, k=k), "fallback_spring"


def layout_spectral(G: nx.DiGraph) -> Dict:
    H = to_undirected_weighted(G)
    return nx.spectral_layout(H)


def layout_mds(G: nx.DiGraph) -> Dict:
    H = to_undirected_weighted(G)
    nodes = list(H.nodes())
    n = len(nodes)
    if n == 0:
        return {}

    dist = dict(nx.all_pairs_shortest_path_length(H))
    big = float(max(5, n))
    D = np.zeros((n, n), dtype=float)
    for i, a in enumerate(nodes):
        for j, b in enumerate(nodes):
            if a == b:
                D[i, j] = 0.0
            else:
                D[i, j] = dist.get(a, {}).get(b, big)

    try:
        from sklearn.manifold import MDS
        mds = MDS(
            n_components=2,
            dissimilarity="precomputed",
            random_state=SEED,
            n_init=1,
            max_iter=300,
        )
        X = mds.fit_transform(D)
    except Exception:
        J = np.eye(n) - np.ones((n, n)) / n
        B = -0.5 * J @ (D ** 2) @ J
        eigvals, eigvecs = np.linalg.eigh(B)
        idx = np.argsort(eigvals)[::-1]
        eigvals = eigvals[idx]
        eigvecs = eigvecs[:, idx]
        L = np.diag(np.sqrt(np.maximum(eigvals[:2], 0.0)))
        X = eigvecs[:, :2] @ L

    pos = {nodes[i]: (float(X[i, 0]), float(X[i, 1])) for i in range(n)}
    return pos


# =========================
# Drawing
# =========================
def draw_save(G: nx.DiGraph, pos: Dict, out_path: str, title: str) -> None:
    ensure_dir(os.path.dirname(out_path))
    pos = normalize_pos(G, pos)

    n = G.number_of_nodes()
    labels = WITH_LABELS and (n <= MAX_LABEL_NODES)

    plt.figure(figsize=FIGSIZE, dpi=DPI)
    plt.title(title)

    nx.draw_networkx_nodes(G, pos, node_size=NODE_SIZE)
    nx.draw_networkx_edges(G, pos, width=EDGE_WIDTH, arrows=True, arrowsize=ARROWSIZE)

    if labels:
        nx.draw_networkx_labels(G, pos, font_size=LABEL_FONT_SIZE)

    plt.axis("off")
    plt.tight_layout()
    plt.savefig(out_path)
    plt.close()


def render_10_layouts(gexf_path: str) -> None:
    alg = alg_name_from_filename(gexf_path)
    G = load_gexf_as_digraph(gexf_path)

    out_dir = os.path.join(OUT_BASE, alg)
    ensure_dir(out_dir)

    print(f"[RUN] {alg}: nodes={G.number_of_nodes()} edges={G.number_of_edges()}")

    # 1) Spring / FR
    draw_save(G, layout_spring_fr(G), os.path.join(out_dir, "01_spring_fr.png"),
              f"{alg} | spring (Fruchterman–Reingold)")

    # 2) Kamada–Kawai
    draw_save(G, layout_kamada_kawai(G), os.path.join(out_dir, "02_kamada_kawai.png"),
              f"{alg} | kamada-kawai")

    # 3) Hierarchical
    pos, tag = layout_hierarchical(G)
    draw_save(G, pos, os.path.join(out_dir, f"03_hierarchical_{tag}.png"),
              f"{alg} | hierarchical ({tag})")

    # 4) Tree
    pos, tag = layout_tree(G)
    draw_save(G, pos, os.path.join(out_dir, f"04_tree_{tag}.png"),
              f"{alg} | tree ({tag})")

    # 5) Circular
    draw_save(G, layout_circular(G), os.path.join(out_dir, "05_circular.png"),
              f"{alg} | circular")

    # 6) Shell / Concentric
    draw_save(G, layout_shell_concentric(G), os.path.join(out_dir, "06_shell_concentric.png"),
              f"{alg} | shell (concentric)")

    # 7) ForceAtlas2
    pos, tag = layout_forceatlas2(G)
    draw_save(G, pos, os.path.join(out_dir, f"07_forceatlas2_{tag}.png"),
              f"{alg} | ForceAtlas2 ({tag})")

    # 8) Yifan Hu
    pos, tag = layout_yifan_hu(G)
    draw_save(G, pos, os.path.join(out_dir, f"08_yifanhu_{tag}.png"),
              f"{alg} | Yifan Hu ({tag})")

    # 9) Spectral
    draw_save(G, layout_spectral(G), os.path.join(out_dir, "09_spectral.png"),
              f"{alg} | spectral")

    # 10) MDS
    draw_save(G, layout_mds(G), os.path.join(out_dir, "10_mds.png"),
              f"{alg} | MDS")

    print(f"[DONE] {alg}: saved -> {out_dir}")


def main():
    gexf_files = sorted(glob.glob(os.path.join(BASE_DIR, "graph_*.gexf")))
    if not gexf_files:
        raise FileNotFoundError(f"No graph_*.gexf found under: {BASE_DIR}")

    for p in gexf_files:
        render_10_layouts(p)


# 주피터에서는 아래 한 줄만 실행해도 됩니다.
# main()
if __name__ == "__main__":
    main()


[RUN] GES: nodes=14 edges=55
[DONE] GES: saved -> ./layouts\GES
[RUN] GOLEM: nodes=13 edges=11
[DONE] GOLEM: saved -> ./layouts\GOLEM
[RUN] NOTEARS: nodes=13 edges=29
[DONE] NOTEARS: saved -> ./layouts\NOTEARS
[RUN] PC: nodes=14 edges=28
[DONE] PC: saved -> ./layouts\PC
